# Baseline vs SpecAugment: Results Comparison

This notebook loads **exported test-set metrics** from:

1. **Scientific baseline** — run `03_evaluate_scientific_baseline.ipynb` (or equivalent evaluation) and save metrics to JSON.
2. **SpecAugment** — run `02_train_scientific_specaugment.ipynb` (after `trainer.predict(test_dataset)`) or a dedicated evaluation notebook, and save the same JSON shape.

It builds side-by-side tables, **absolute and relative changes** (SpecAugment minus baseline), and comparison plots.


## Metric export format

Each file should contain a **flat JSON object** of metric names to floats, as returned by Hugging Face `Trainer.predict` (keys often look like `test_accuracy`, `test_f1_macro`, …). The loader strips the `test_` prefix for plotting.

Example (save from your evaluation cell):

```python
import json
from pathlib import Path
out = Path("results") / "baseline_test_metrics.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps({k: float(v) for k, v in test_pred.metrics.items()}, indent=2))
```


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


In [ ]:
# Paths to JSON exports (change if you store results elsewhere)
BASELINE_METRICS_PATH = Path("results/baseline_test_metrics.json")
SPECAUGMENT_METRICS_PATH = Path("results/specaugment_test_metrics.json")

FIGURE_DIR = Path("results") / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Optional: Colab / Drive layout (uncomment and adjust if needed)
# BASELINE_METRICS_PATH = Path("/content/drive/MyDrive/.../baseline_test_metrics.json")
# SPECAUGMENT_METRICS_PATH = Path("/content/drive/MyDrive/.../specaugment_test_metrics.json")


In [ ]:
def load_metrics_json(path: Path) -> dict[str, float]:
    data = json.loads(path.read_text(encoding="utf-8"))
    return {k: float(v) for k, v in data.items()}


def normalize_keys(raw: dict[str, float]) -> dict[str, float]:
    """Strip leading 'test_' so baseline and SpecAugment line up."""
    out: dict[str, float] = {}
    for k, v in raw.items():
        nk = k[5:] if k.startswith("test_") else k
        out[nk] = v
    return out


PLACEHOLDER_BASELINE = {
    "accuracy": 0.72,
    "f1_macro": 0.70,
    "precision_macro": 0.69,
    "recall_macro": 0.71,
    "f1_weighted": 0.72,
    "f1_nodementia": 0.74,
    "f1_dementia": 0.66,
}

PLACEHOLDER_SPECAUGMENT = {
    "accuracy": 0.75,
    "f1_macro": 0.73,
    "precision_macro": 0.72,
    "recall_macro": 0.74,
    "f1_weighted": 0.75,
    "f1_nodementia": 0.76,
    "f1_dementia": 0.70,
}


def load_or_placeholder(path: Path, which: str) -> tuple[dict[str, float], bool]:
    if path.is_file():
        return normalize_keys(load_metrics_json(path)), False
    print(f"Warning: {path} not found — using placeholder {which} metrics for demo.")
    if which == "baseline":
        return dict(PLACEHOLDER_BASELINE), True
    return dict(PLACEHOLDER_SPECAUGMENT), True


baseline_m, baseline_ph = load_or_placeholder(BASELINE_METRICS_PATH, "baseline")
spec_m, spec_ph = load_or_placeholder(SPECAUGMENT_METRICS_PATH, "specaugment")


In [ ]:
# Core classification metrics (intersection of keys present in both runs)
PREFERRED_ORDER = [
    "accuracy",
    "f1_macro",
    "precision_macro",
    "recall_macro",
    "f1_weighted",
    "f1_nodementia",
    "f1_dementia",
]

common_keys = [k for k in PREFERRED_ORDER if k in baseline_m and k in spec_m]
extra_baseline = sorted(set(baseline_m) - set(common_keys) - {"loss", "eval_loss", "test_loss", "loss"})
extra_spec = sorted(set(spec_m) - set(common_keys) - {"loss", "eval_loss", "test_loss", "loss"})
if extra_baseline or extra_spec:
    print("Note: non-overlapping keys (excluded from main compare):", extra_baseline, extra_spec)

rows = []
for k in common_keys:
    b, s = baseline_m[k], spec_m[k]
    delta = s - b
    pct = (delta / b * 100.0) if b != 0 else float("nan")
    rows.append(
        {"metric": k, "baseline": b, "specaugment": s, "delta": delta, "pct_change_vs_baseline": pct}
    )

compare_df = pd.DataFrame(rows)
compare_df


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(common_keys))
w = 0.36
ax.bar(x - w / 2, compare_df["baseline"], width=w, label="Baseline", color="#4C72B0")
ax.bar(x + w / 2, compare_df["specaugment"], width=w, label="SpecAugment", color="#DD8452")
ax.set_xticks(x)
ax.set_xticklabels(common_keys, rotation=25, ha="right")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.set_title("Test metrics: baseline vs SpecAugment")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "baseline_vs_specaugment_grouped_bars.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
macro_block = [k for k in ("accuracy", "f1_macro", "precision_macro", "recall_macro", "f1_weighted") if k in common_keys]
if macro_block:
    fig, ax = plt.subplots(figsize=(8, 4))
    deltas = compare_df.set_index("metric").loc[macro_block, "delta"].values
    colors = ["#55A868" if d >= 0 else "#C44E52" for d in deltas]
    ax.barh(macro_block, deltas, color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Change (SpecAugment - baseline)")
    ax.set_title("Absolute metric change on test set")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "baseline_vs_specaugment_delta_absolute.png", dpi=200, bbox_inches="tight")
    plt.show()


In [ ]:
class_f1 = [k for k in ("f1_nodementia", "f1_dementia") if k in common_keys]
if class_f1:
    fig, ax = plt.subplots(figsize=(6, 4))
    x = np.arange(len(class_f1))
    w = 0.36
    bvals = [baseline_m[k] for k in class_f1]
    svals = [spec_m[k] for k in class_f1]
    ax.bar(x - w / 2, bvals, width=w, label="Baseline", color="#4C72B0")
    ax.bar(x + w / 2, svals, width=w, label="SpecAugment", color="#DD8452")
    ax.set_xticks(x)
    labels = ["No dementia (F1)", "Dementia (F1)"] if class_f1 == ["f1_nodementia", "f1_dementia"] else class_f1
    ax.set_xticklabels(labels)
    ax.set_ylabel("F1")
    ax.set_ylim(0, 1.05)
    ax.set_title("Per-class F1 on test set")
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "baseline_vs_specaugment_per_class_f1.png", dpi=200, bbox_inches="tight")
    plt.show()


In [ ]:
if macro_block:
    fig, ax = plt.subplots(figsize=(8, 4))
    pct = compare_df.set_index("metric").loc[macro_block, "pct_change_vs_baseline"].values
    pct_clean = np.nan_to_num(pct, nan=0.0)
    colors = ["#55A868" if d >= 0 else "#C44E52" for d in pct_clean]
    ax.barh(macro_block, pct_clean, color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Percent change vs baseline (%)")
    ax.set_title("Relative change (NaN coerced to 0 only for plotting if baseline was 0)")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "baseline_vs_specaugment_delta_percent.png", dpi=200, bbox_inches="tight")
    plt.show()


In [ ]:
compare_df.to_csv(FIGURE_DIR / "baseline_vs_specaugment_metrics.csv", index=False)
print("Saved", FIGURE_DIR / "baseline_vs_specaugment_metrics.csv")
